# Stage 00b — Descriptions CSV + Positional Figure Matching

**What this notebook does:**

1. Exports all *Brief Description of the Drawings* texts from the PatSeer Excel
   into a single `data/descriptions.csv` (replaces the old per-patent `.txt` files).

2. Assigns a figure label to every downloaded image using **pure positional
   matching** — no OCR model, no external service.

The matching logic is:
1. Load all files for a patent from `raw/{patent_id}/` in order: `img*` → `D*` → `FAT*`
2. For `D` and `FAT` files, detect horizontal/vertical whitespace bands (in-memory) and
   count sub-figures without saving intermediate crops
3. Parse the description text → ordered figure key list (FIG. 1, FIG. 2A, …)
4. If **crop count == description count** → positional 1-to-1 match → `_F` names
5. If **counts differ** → flag everything `_Fu` (needs human review via Stage 01)

**Output naming:**

| Before | After | Meaning |
|--------|-------|---------|
| `US…_img003.png` | `US…_img003_F002B.png` | matched to FIG. 2B |
| `US…_img003.png` | `US…_img003_Fu001.png` | count mismatch — needs review |
| `US…_D00005.png` | `US…_D00005_crop01_F001.png` | first crop of a split D sheet |
| `US…_D00005.png` | kept as-is | original D file always preserved |

**Where it fits in the pipeline:**
```
00a  (PatSeer download)
 ↓
00b  ←  YOU ARE HERE  (descriptions CSV + positional matching → _F / _Fu labels)
 ↓
01   (optional OCR fallback for _Fu files that need review)
 ↓
02   (pad + resize to 518×518)
```

---

| Cell | What it does |
|------|--------------|
| 1 | Load config, Excel index, and figure_matcher functions |
| 2 | Save `data/descriptions.csv` — all brief-description texts in one CSV |
| 3 | Dry-run on one patent — shows split detection, crop list, rename preview; **no files changed** |
| 4 | Full run — match and rename all patents in `raw/` |
| 5 | Print matching report (_F count, _Fu count, originals kept) |


In [1]:
import sys, importlib.util, re
from pathlib import Path

# Walk upward from cwd until we find the repo root (has both src/ and config.yaml).
# This handles VS Code running notebooks with cwd = workspace root.
_cwd = Path().resolve()
repo_root = None
for _candidate in [_cwd, *_cwd.parents]:
    if (_candidate / "src").exists() and (_candidate / "config.yaml").exists():
        repo_root = _candidate
        break
if repo_root is None:
    raise RuntimeError(f"Cannot find repo root from {_cwd}. Run from inside Patent-Labelling-Tools.")
src_dir = repo_root / "src"
print(f"repo_root : {repo_root}")
print(f"src_dir   : {src_dir}")

_cl_path = src_dir / "config_loader.py"
_cl_spec = importlib.util.spec_from_file_location("config_loader", _cl_path)
_cl_mod  = importlib.util.module_from_spec(_cl_spec)
sys.modules["config_loader"] = _cl_mod
_cl_spec.loader.exec_module(_cl_mod)
load_config = _cl_mod.load_config

for p in [str(repo_root), str(src_dir)]:
    while p in sys.path:
        sys.path.remove(p)
sys.path.insert(0, str(repo_root))
sys.path.insert(0, str(src_dir))

# Force reimport in case of a previous failed attempt
for _mod in ["figure_matcher", "extractor"]:
    sys.modules.pop(_mod, None)

from figure_matcher import (
    parse_description_figures,
    detect_split_needed,
    match_positionally,
    rename_matched_files,
)
from extractor import load_patseer_excel, save_descriptions_csv

cfg         = load_config()
excel_index = load_patseer_excel(cfg["paths"]["patseer_excel"])
raw_dir     = cfg["paths"]["raw_images"]
print(f"Loaded {len(excel_index)} patents from Excel.")


def _patent_core(patent_id: str) -> str:
    """Strip country code + kind code; pad compact 6-digit US serial to 7."""
    m = re.match(r"^[A-Z]{2,}(\d+)", patent_id, re.IGNORECASE)
    if not m:
        return patent_id
    num = m.group(1)
    m2 = re.match(r"^((?:19|20)\d{2})(\d{6})$", num)
    if m2:
        num = m2.group(1) + "0" + m2.group(2)
    return num


def _build_folder_map(raw_dir: Path) -> dict:
    """core → actual folder Path, so any patent ID variant resolves correctly."""
    return {_patent_core(d.name): d for d in raw_dir.iterdir() if d.is_dir()}


repo_root : /home/vasco/Vasco Workspace/Tese_Vasco_Lnx/Patent-Labelling-Tools
src_dir   : /home/vasco/Vasco Workspace/Tese_Vasco_Lnx/Patent-Labelling-Tools/src


/home/vasco/anaconda3/envs/doclayout_yolo/lib/python3.10/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


PatSeer Excel: 1639__dataset_08_06_26.xlsx  (1639 rows, 102 columns)
Columns:
  [  0] 'Record Number'
  [  1] 'Title'
  [  2] 'Abstract'
  [  3] 'Description of Drawings'
  [  4] 'CPC'
  [  5] 'PDF Link'
  [  6] 'Record Type'
  [  7] 'Publication/Issue Date'
  [  8] 'Filing/Application Date'
  [  9] 'Estimated Expiry Date'
  [ 10] 'EFAM Earliest Priority Date'
  [ 11] 'EFAM Earliest Publication Date'
  [ 12] 'Priority Details'
  [ 13] 'Priority Dates (All)'
  [ 14] 'Application No.'
  [ 15] 'Priority Country Code'
  [ 16] 'Priority Year'
  [ 17] 'Register Legal Status'
  [ 18] 'Register Legal Status Date'
  [ 19] 'Summary of Invention'
  [ 20] 'Designated States'
  [ 21] 'Active in Designated States'
  [ 22] 'Field of search'
  [ 23] 'Industry'
  [ 24] 'Tech Domain'
  [ 25] 'Tech Sub Domain'
  [ 26] 'Description'
  [ 27] 'Claims'
  [ 28] 'Number Of Claims'
  [ 29] 'No. of Independent Claims'
  [ 30] 'Independent Claims'
  [ 31] 'First Claim'
  [ 32] 'Advantages of Invention'
  [ 33] 'N

In [2]:
# ─── Save brief-description-of-drawings CSV ─────────────────────────────────
# Reads the 'description_of_drawings' column already loaded from the Excel
# and writes one CSV: data/descriptions.csv  (patent_id, description_of_drawings)
# This replaces the individual text/{patent_id}.txt files from the old pipeline.

data_dir  = cfg["paths"]["data"]
csv_out   = save_descriptions_csv(excel_index, data_dir / "descriptions.csv")

n_total = len(excel_index)
n_filled = sum(1 for m in excel_index.values() if m.get("description_of_drawings"))
print(f"Saved {n_total} rows → {csv_out}")
print(f"  {n_filled} patents have a description  |  {n_total - n_filled} are empty")


Saved 1639 rows → /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_DS/data/descriptions.csv
  1282 patents have a description  |  357 are empty


In [3]:
# ─── Dry-run on ONE patent — positional matching preview, NO files changed ─────
TEST_PATENT = "US2015014475A1"   # any Excel key or folder name

folder_map = _build_folder_map(raw_dir)
img_dir    = folder_map.get(_patent_core(TEST_PATENT))

if img_dir is None:
    print(f"\u26a0  No folder found for {TEST_PATENT}")
    print(f"   Available: {sorted(folder_map.keys())[:10]}")
else:
    actual_id = img_dir.name
    desc      = excel_index.get(TEST_PATENT, {}).get("description_of_drawings", "")

    # ── Description figures ───────────────────────────────────────────────────
    fig_keys = parse_description_figures(desc)
    print(f"Patent     : {actual_id}")
    print(f"Description figures ({len(fig_keys)}): {fig_keys}")
    print()

    # ── File inventory ────────────────────────────────────────────────────────
    _skip = re.compile(r"_(?:F\d{3}|Fu\d{3}|crop\d{2})", re.IGNORECASE)
    img_files = [p for p in sorted(img_dir.glob(f"{actual_id}_img*.png")) if not _skip.search(p.stem)]
    d_files   = [p for p in sorted(img_dir.glob(f"{actual_id}_D*.png"))   if not _skip.search(p.stem)]
    fat_files = [p for p in sorted(img_dir.glob(f"{actual_id}_FAT*.png")) if not _skip.search(p.stem)]
    print(f"File inventory: img={len(img_files)}  D={len(d_files)}  FAT={len(fat_files)}")
    print()

    # ── Split detection per file ──────────────────────────────────────────────
    print("Split detection:")
    for f in img_files + d_files + fat_files:
        n = detect_split_needed(f)
        tag = f"→ {n} sub-figures" if n > 1 else "→ no split"
        print(f"  {f.name:<55}  {tag}")
    print()

    # ── Count validation ──────────────────────────────────────────────────────
    matches  = match_positionally(actual_id, img_dir, desc)
    n_matchable = sum(1 for m in matches if m["source_type"] != "FAT")
    if n_matchable == len(fig_keys) and len(fig_keys) > 0:
        print(f"Count validation: MATCH ({n_matchable} crops == {len(fig_keys)} description keys)")
    else:
        print(f"Count validation: MISMATCH ({n_matchable} crops \u2260 {len(fig_keys)} description keys)")
    print()

    # ── Rename preview (no actual rename) ─────────────────────────────────────
    print("Rename preview:")
    print(f"  {'Source file':<50}  {'Output filename'}")
    print(f"  {'-'*50}  {'-'*50}")
    for m in matches:
        tag = "\u2713" if not m["needs_review"] else "\u26a0"
        print(f"  {tag} {m['source_file']:<48}  \u2192  {m['output_filename']}")


Patent     : US20150014475PAFP
Description figures (12): ['1', '2A', '3A', '3B', '3C', '4A', '5', '6', '7A', '8A', '9', '10']

File inventory: img=11  D=0  FAT=0

Split detection:
  US20150014475PAFP_img1.png                               → 10 sub-figures
  US20150014475PAFP_img10.png                              → 19 sub-figures
  US20150014475PAFP_img11.png                              → 9 sub-figures
  US20150014475PAFP_img2.png                               → 25 sub-figures
  US20150014475PAFP_img3.png                               → 28 sub-figures
  US20150014475PAFP_img4.png                               → 23 sub-figures
  US20150014475PAFP_img5.png                               → 27 sub-figures
  US20150014475PAFP_img6.png                               → 19 sub-figures
  US20150014475PAFP_img7.png                               → 24 sub-figures
  US20150014475PAFP_img8.png                               → 9 sub-figures
  US20150014475PAFP_img9.png                               → 3

In [ ]:
# ─── Full run — positional matching + rename all patents ──────────────────────
# Uses the same patent ordering as 00_image_extractor: dedup → group → display_order.
# WARNING: run_grouping downloads ~500 MB (PatentSBERTa) on first call.
# If you want a simpler run without grouping, replace patent_ids with
#   patent_ids = list(excel_index.keys())

import sys; sys.path.insert(0, str(repo_root))   # ensure src/ on path
import pandas as pd
from deduplicator import run_deduplication
from grouper import run_grouping

raw_df          = pd.read_excel(cfg["paths"]["patseer_excel"], dtype=str)
deduplicated_df, _ = run_deduplication(raw_df, cfg)
grouped_df      = run_grouping(deduplicated_df, cfg)

_PUB_VARIANTS = ["Publication Number", "Pub. No.", "Patent Number", "Record Number"]
_pub_col = next((c for c in _PUB_VARIANTS if c in grouped_df.columns), None)
if _pub_col is None:
    raise KeyError(f"No pub-number column found. Got: {list(grouped_df.columns[:10])}")

patent_ids = grouped_df.sort_values("display_order")[_pub_col].tolist()
print(f"Processing queue: {len(patent_ids)} patents (deduped, grouped ordering)")
print()

folder_map = _build_folder_map(raw_dir)
results    = {}
errors     = []

for patent_id in patent_ids:
    img_dir = folder_map.get(_patent_core(patent_id))
    if img_dir is None:
        continue

    actual_id = img_dir.name
    desc      = excel_index.get(patent_id, {}).get("description_of_drawings", "")

    try:
        matches = match_positionally(actual_id, img_dir, desc)
        summary = rename_matched_files(matches, img_dir)
        results[patent_id] = summary
        alias = f" ({patent_id})" if actual_id != patent_id else ""
        print(f"  {actual_id}{alias}  _F={summary['renamed_F']}  _Fu={summary['renamed_Fu']}")
    except Exception as exc:
        errors.append((patent_id, str(exc)))
        print(f"  \u2717 {patent_id}: {exc}")

print(f"\nDone. {len(results)} patents processed, {len(errors)} errors.")


In [ ]:
# ─── Matching report ─────────────────────────────────────────────────────────
if not results:
    print("No results yet — run Cell 4 first.")
else:
    total    = len(results)
    perfect  = sum(1 for s in results.values() if s["renamed_Fu"] == 0)
    needs_rv = total - perfect
    total_F  = sum(s["renamed_F"]      for s in results.values())
    total_Fu = sum(s["renamed_Fu"]     for s in results.values())
    kept     = sum(s["kept_originals"] for s in results.values())
    err      = sum(s["errors"]         for s in results.values()) + len(errors)

    def _pct(n):
        return f"{n / total * 100:.0f}%" if total else "N/A"

    print("\u2550" * 38)
    print("Figure Matching Report")
    print("\u2550" * 38)
    print(f"Patents processed       : {total}")
    print(f"Perfect matches (_F)    : {perfect}  ({_pct(perfect)})")
    print(f"Needs review (_Fu)      : {needs_rv}  ({_pct(needs_rv)})")
    print(f"D/FAT originals kept    : {kept}")
    print(f"Total figures renamed   : {total_F + total_Fu}")
    print(f"  - labeled   (_F)      : {total_F}")
    print(f"  - unlabeled (_Fu)     : {total_Fu}")
    print(f"Run errors              : {err}")
    print("\u2550" * 38)

    if errors:
        print("\nErrors:")
        for pid, e in errors:
            print(f"  {pid}: {e}")
